## 📊 Main Analysis: Climate Discourse Metrics

This section loads all processed reports and calculates key metrics:

**Core Metrics:**
- **Volume**: Percentage of report content that's climate-related
- **Specificity**: How concrete vs. vague the climate claims are (0-100)
- **Commitment**: How action-oriented the language is (0-100)  
- **Talk Score**: Composite metric = 20% volume + 40% specificity + 40% commitment
- **Sentiment**: Opportunity / Neutral / Risk framing (confidence-weighted)

**Data Processing:**
1. Load all `*_bert.json` files from cache
2. Exclude factsheets/highlights (too short, distort metrics)
3. Aggregate to company-year level
4. Calculate weighted averages using detector confidence scores

**Outputs:**
- `slide_main.png`: Volume, Specificity, Commitment trends
- `slide_sentiment_trend.png`: Sentiment evolution
- `talk_score_trend.png`: Talk score with year-over-year delta
- `per_company_*.png`: Company-level breakdowns

In [ ]:
# =============================================================================
# STEEL INDUSTRY CLIMATE REPORT ANALYSIS - FINAL
# =============================================================================

from collections import Counter
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# === CONFIG ===
DATA_DIR = './cache'
OUTPUT_DIR = './out'
EXCLUDE_PATTERNS = [
    'factsheet',
    'fact_sheet',
    'faktensheets',
    'faktenblaetter',
    'faktenblatt',
    'daten_und_fakten',
    'zahlen_und_fakten',
]
EXCLUDE_2025 = True  # Incomplete year

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

# --- Load bert.json files ---


def load_and_process():
    data_path = Path(DATA_DIR)
    bert_files = [f for f in data_path.glob('*_bert.json')
                  if not any(ex.lower() in f.name.lower() for ex in EXCLUDE_PATTERNS)]

    rows = []
    for fp in bert_files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
            chunks = data.get('chunks', [])
            if not chunks:
                continue

            n = len(chunks)
            detector = np.array([c.get('detector_score', 0) for c in chunks])
            weights = detector if detector.sum() > 0 else np.ones(n)

            spec_labels = [c.get('specificity_label', 'non') for c in chunks]
            sent_labels = [c.get('sentiment_label', 'neutral') for c in chunks]
            comm_labels = [c.get('commitment_label', 'no') for c in chunks]

            # Confidence-weighted sentiment percentages
            # Each chunk's sentiment vote is weighted by detector confidence
            opp_weights = sum(w for w, l in zip(
                weights, sent_labels) if l == 'opportunity')
            risk_weights = sum(w for w, l in zip(
                weights, sent_labels) if l == 'risk')
            neutral_weights = sum(w for w, l in zip(
                weights, sent_labels) if l == 'neutral')
            total_weights = weights.sum()

            rows.append({
                'company': data.get('company', 'Unknown'),
                'company_id': str(data.get('company_id', 'unknown')),
                'year': int(data.get('year', 0)),
                'is_translated': data.get('source', 'original') == 'translated',
                'climate_chunks': data.get('climate_chunks', n),
                'climate_pct': data.get('kept_percentage', 100.0),
                'specificity_w': np.average([c.get('specificity_score', 0) for c in chunks], weights=weights),
                'commitment_w': np.average([c.get('commitment_score', 0) for c in chunks], weights=weights),
                # Confidence-weighted sentiment percentages
                'opportunity_pct': (opp_weights / total_weights * 100) if total_weights > 0 else 0,
                'risk_pct': (risk_weights / total_weights * 100) if total_weights > 0 else 0,
                'neutral_pct': (neutral_weights / total_weights * 100) if total_weights > 0 else 0,
            })
        except:
            pass

    df = pd.DataFrame(rows)
    if EXCLUDE_2025:
        df = df[df['year'] < 2025]

    # Aggregate to company-year
    cy = df.groupby(['company_id', 'year']).agg({
        'company': 'first', 'is_translated': 'any', 'climate_chunks': 'sum',
        'climate_pct': 'mean', 'specificity_w': 'mean', 'commitment_w': 'mean',
        'opportunity_pct': 'mean', 'risk_pct': 'mean', 'neutral_pct': 'mean',
    }).reset_index()

    # Talk Score: 20% volume + 40% specificity + 40% commitment (quality-focused)
    cy['talk_score'] = ((cy['climate_pct']/100)*0.20 +
                        cy['specificity_w']*0.40 + cy['commitment_w']*0.40) * 100

    # Sentiment balances
    cy['sent_v1'] = cy['opportunity_pct'] - cy['risk_pct']
    cy['sent_v2'] = (cy['opportunity_pct'] +
                     cy['neutral_pct']) - cy['risk_pct']

    # YoY deltas
    cy = cy.sort_values(['company_id', 'year'])
    cy['talk_score_delta'] = cy.groupby('company_id')['talk_score'].diff()

    return df, cy


report_df, cy_df = load_and_process()
print(
    f"✅ Loaded: {len(report_df)} reports, {cy_df['company'].nunique()} companies, {cy_df['year'].min()}-{cy_df['year'].max()}")

# %% CELL 2: ALL PLOTS


def save(name):
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{name}', dpi=200, bbox_inches='tight')
    print(f"   ✓ {name}")
    plt.show()


yearly = cy_df.groupby('year').agg({
    'climate_pct': 'mean', 'specificity_w': 'mean', 'commitment_w': 'mean',
    'talk_score': ['mean', 'std'], 'talk_score_delta': 'mean',
    'opportunity_pct': 'mean', 'risk_pct': 'mean', 'neutral_pct': 'mean',
    'company_id': 'nunique',
}).reset_index()
yearly.columns = ['year', 'volume', 'specificity', 'commitment', 'talk_mean', 'talk_std',
                  'talk_delta', 'opp', 'risk', 'neutral', 'n_companies']

# === 1. MAIN SLIDE: Volume, Specificity, Commitment ===
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(yearly['year'], yearly['volume'], '#7B1FA2', lw=3.5,
        marker='o', ms=11, label='Volume (% climate content)')
ax.plot(yearly['year'], yearly['specificity']*100, '#1565C0',
        lw=3.5, marker='s', ms=11, label='Specificity')
ax.plot(yearly['year'], yearly['commitment']*100, '#2E7D32',
        lw=3.5, marker='^', ms=11, label='Commitment')
ax.set_xlabel('Year', fontsize=14)
ax.set_ylabel('Score (0-100)', fontsize=14)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.set_title('Climate Reporting in the Steel Industry',
             fontsize=18, fontweight='bold')
ax.legend(loc='lower right', fontsize=11, frameon=True, fancybox=True,
          shadow=True, facecolor='#F5F5F5', edgecolor='gray')
save('slide_main.png')

# === 2. SENTIMENT TREND (Separate Lines) ===
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly['year'], yearly['opp'], 'g-o', lw=3, ms=9, label='Opportunity')
ax.plot(yearly['year'], yearly['neutral'], 'gray',
        ls='-', marker='s', lw=3, ms=9, label='Neutral')
ax.plot(yearly['year'], yearly['risk'], 'r-^', lw=3, ms=9, label='Risk')
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('% of Climate Chunks', fontsize=13)
ax.set_title('Sentiment Trend', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=11, frameon=True, fancybox=True,
          shadow=True, facecolor='#F5F5F5', edgecolor='gray')
save('slide_sentiment_trend.png')

# === 3. TALK SCORE TREND + DELTA ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]
ax1.plot(yearly['year'], yearly['talk_mean'],
         '#1976D2', lw=3, marker='o', ms=10)
ax1.fill_between(yearly['year'], yearly['talk_mean']-yearly['talk_std'],
                 yearly['talk_mean']+yearly['talk_std'], alpha=0.2)
ax1.set_xlabel('Year')
ax1.set_ylabel('Talk Score (0-100)')
ax1.set_title(
    'Talk Score Trend\n(20% volume + 40% specificity + 40% commitment)', fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
colors = ['green' if d > 0 else 'red' for d in yearly['talk_delta'].fillna(0)]
ax2.bar(yearly['year'], yearly['talk_delta'].fillna(
    0), color=colors, alpha=0.7, edgecolor='white')
ax2.axhline(0, color='black', lw=1)
ax2.set_xlabel('Year')
ax2.set_ylabel('YoY Change')
ax2.set_title('Talk Score YoY Delta', fontweight='bold')
ax2.grid(True, alpha=0.3)
save('talk_score_trend.png')

# === 4. TALK SCORE PER COMPANY ===
company_stats = cy_df.groupby('company').agg({'talk_score': [
    'mean', 'last'], 'talk_score_delta': 'mean', 'year': 'count'}).reset_index()
company_stats.columns = ['company', 'talk_mean',
                         'talk_latest', 'talk_delta_avg', 'n_years']
company_stats = company_stats.sort_values('talk_latest')

fig, ax = plt.subplots(figsize=(10, max(6, len(company_stats)*0.4)))
colors = plt.cm.RdYlGn(company_stats['talk_latest']/100)
ax.barh(company_stats['company'], company_stats['talk_latest'],
        color=colors, edgecolor='white')
for i, (_, r) in enumerate(company_stats.iterrows()):
    ax.text(r['talk_latest']+1, i,
            f'{r["talk_latest"]:.0f}', va='center', fontsize=9)
ax.set_xlabel('Talk Score (Latest Year)')
ax.set_xlim(0, 105)
ax.set_title('Talk Score by Company', fontweight='bold')
save('talk_score_per_company.png')

# === 5. PER-COMPANY: Volume, Specificity, Commitment + Talk Score ===
companies = sorted(cy_df['company'].unique())
n_cols, n_rows = 3, int(np.ceil(len(companies)/3))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(
    5*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()

for i, company in enumerate(companies):
    ax = axes[i]
    cd = cy_df[cy_df['company'] == company].sort_values('year')
    ax.bar(cd['year'], cd['climate_pct'], color='lightgray',
           alpha=0.5, label='Volume', width=0.7)
    ax.plot(cd['year'], cd['specificity_w']*100,
            'b-s', ms=5, lw=1.5, label='Specificity')
    ax.plot(cd['year'], cd['commitment_w']*100,
            'g-^', ms=5, lw=1.5, label='Commitment')
    ax.plot(cd['year'], cd['talk_score'], 'k-o',
            ms=6, lw=2, label='Talk Score')
    ax.set_title(company, fontweight='bold', fontsize=10)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, loc='upper left')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Per-Company: Components + Talk Score',
             fontsize=14, fontweight='bold', y=1.01)
save('per_company_components.png')

# === 6. PER-COMPANY SENTIMENT (STACKED AREA) ===
fig, axes = plt.subplots(n_rows, n_cols, figsize=(
    5*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()

for i, company in enumerate(companies):
    ax = axes[i]
    cd = cy_df[cy_df['company'] == company].sort_values('year')

    # Stacked area: Opportunity (bottom), Neutral (middle), Risk (top)
    ax.fill_between(cd['year'], 0, cd['opportunity_pct'],
                    color='green', alpha=0.7, label='Opportunity')
    ax.fill_between(cd['year'], cd['opportunity_pct'], cd['opportunity_pct']+cd['neutral_pct'],
                    color='gray', alpha=0.7, label='Neutral')
    ax.fill_between(cd['year'], cd['opportunity_pct']+cd['neutral_pct'], 100,
                    color='red', alpha=0.7, label='Risk')

    ax.set_title(company, fontweight='bold', fontsize=10)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Per-Company Sentiment Composition',
             fontsize=14, fontweight='bold', y=1.01)
save('per_company_sentiment.png')

# === 7. ALL COMPANIES SENTIMENT TRAJECTORY ===
fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.tab20(np.linspace(0, 1, len(companies)))
for i, company in enumerate(companies):
    cd = cy_df[cy_df['company'] == company].sort_values('year')
    ax.plot(cd['year'], cd['sent_v2'], '-o',
            color=colors[i], lw=1.5, ms=5, alpha=0.6)
    if len(cd) > 0:
        ax.annotate(company, (cd['year'].iloc[-1]+0.1,
                    cd['sent_v2'].iloc[-1]), fontsize=7, color=colors[i])
# Industry average
yearly_sent = cy_df.groupby('year')['sent_v2'].mean()
ax.plot(yearly_sent.index, yearly_sent.values,
        'k-', lw=4, label='Industry Avg', zorder=10)
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Sentiment: (Opp+Neutral) - Risk', fontsize=12)
ax.set_title('Sentiment Trajectory by Company', fontsize=14, fontweight='bold')
ax.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1]+1.5)
ax.grid(True, alpha=0.3)
save('sentiment_all_companies.png')

# === SUMMARY STATS ===
n_reports = len(report_df)
n_chunks = int(cy_df['climate_chunks'].sum())
n_companies = cy_df['company'].nunique()
year_range = f"{cy_df['year'].min()}-{cy_df['year'].max()}"

print(f"""
{'='*60}
📊 SUMMARY FOR SLIDES
{'='*60}
Data: {n_reports} reports · {n_chunks:,} chunks · {n_companies} companies · {year_range}
Method: ClimateBERT (scores weighted by detector confidence)

Talk Score = 20% volume + 40% specificity + 40% commitment

Key Findings:
• Volume ↑ — more climate content in reports
• Quality → — specificity & commitment remain flat
• Sentiment shift: Opportunity → Neutral → Rising Risk
{'='*60}
""")


def load_all_chunk_texts():
    """Load all chunk texts from bert.json files."""
    data_path = Path(DATA_DIR)
    bert_files = [f for f in data_path.glob('*_bert.json')
                  if not any(ex.lower() in f.name.lower() for ex in EXCLUDE_PATTERNS)]

    all_texts = []
    texts_by_sentiment = {'opportunity': [], 'risk': [], 'neutral': []}
    texts_by_commitment = {'yes': [], 'no': []}

    for fp in bert_files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)

            year = data.get('year', 0)
            if EXCLUDE_2025 and year >= 2025:
                continue

            for chunk in data.get('chunks', []):
                text = chunk.get('text', '')
                if text:
                    all_texts.append(text)

                    # Group by sentiment
                    sent = chunk.get('sentiment_label', 'neutral')
                    if sent in texts_by_sentiment:
                        texts_by_sentiment[sent].append(text)

                    # Group by commitment
                    comm = chunk.get('commitment_label', 'no')
                    if comm in texts_by_commitment:
                        texts_by_commitment[comm].append(text)
        except:
            pass

    return all_texts, texts_by_sentiment, texts_by_commitment


def get_word_frequencies(texts, min_word_len=4, top_n=50):
    """Get word frequencies with grouping under readable representative words."""
    # Common stopwords to exclude
    stopwords = {
        # English stopwords
        'the', 'and', 'for', 'that', 'with', 'are', 'from', 'this', 'was', 'were',
        'been', 'have', 'has', 'will', 'would', 'could', 'should', 'their', 'they',
        'which', 'what', 'when', 'where', 'who', 'how', 'our', 'your', 'more', 'also',
        'than', 'into', 'over', 'such', 'through', 'being', 'between', 'after', 'before',
        'these', 'those', 'other', 'some', 'most', 'about', 'including', 'within',
        # Domain words
        'steel', 'company', 'group', 'year', 'years', 'report', 'reporting', 'annual',
        # Company names (lowercase)
        'arcelormittal', 'arcelor', 'mittal', 'thyssenkrupp', 'thyssen', 'krupp',
        'voestalpine', 'salzgitter', 'ssab', 'outokumpu', 'tata', 'tatasteel',
        'nippon', 'nipponsteeel', 'baosteel', 'baoshan', 'posco', 'nucor',
        'acerinox', 'celsa', 'dillinger', 'sidenor', 'feralpi', 'arvedi',
        'acciaierieditalia', 'acciaieri', 'italia', 'nederland', 'tatasteelnederland',
        'tatasteeluk', 'tatasteeleurope',
    }

    # Word groups: map variants to a single readable representative
    # Key = representative word, Values = all variants that should count as this word
    word_groups = {
        'emission': ['emission', 'emissions', 'emitting', 'emitted', 'emit'],
        'carbon': ['carbon', 'carbons', 'co2', 'dioxide'],
        'energy': ['energy', 'energies', 'energetic'],
        'sustainable': ['sustainable', 'sustainability', 'sustainably'],
        'environment': ['environment', 'environmental', 'environmentally', 'environments'],
        'climate': ['climate', 'climatic', 'climates'],
        'reduction': ['reduction', 'reductions', 'reduce', 'reduced', 'reducing', 'reduces'],
        'target': ['target', 'targets', 'targeted', 'targeting'],
        'goal': ['goal', 'goals'],
        'commitment': ['commitment', 'commitments', 'commit', 'commits', 'committed', 'committing'],
        'production': ['production', 'productions', 'produce', 'produced', 'producing', 'produces', 'productive'],
        'price': ['price','prices'],
        'process': ['process', 'processes', 'processing', 'processed'],
        'technology': ['technology', 'technologies', 'technological', 'tech'],
        'innovation': ['innovation', 'innovations', 'innovative', 'innovate', 'innovating'],
        'investment': ['investment', 'investments', 'invest', 'invested', 'investing', 'investor', 'investors'],
        'development': ['development', 'developments', 'develop', 'developed', 'developing', 'develops'],
        'strategy': ['strategy', 'strategies', 'strategic', 'strategically'],
        'management': ['management', 'manage', 'managed', 'managing', 'manager', 'managers'],
        'performance': ['performance', 'performances', 'perform', 'performed', 'performing'],
        'business': ['business', 'businesses'],
        'operation': ['operation', 'operations', 'operate', 'operated', 'operating', 'operational'],
        'employee': ['employee', 'employees', 'employment', 'employer', 'employers'],
        'safety': ['safety', 'safe', 'safely', 'safer'],
        'risk': ['risk', 'risks', 'risky'],
        'opportunity': ['opportunity', 'opportunities'],
        'challenge': ['challenge', 'challenges', 'challenging', 'challenged'],
        'impact': ['impact', 'impacts', 'impacted', 'impacting'],
        'measure': ['measure', 'measures', 'measured', 'measuring', 'measurement', 'measurements'],
        'improve': ['improve', 'improved', 'improving', 'improvement', 'improvements'],
        'increase': ['increase', 'increased', 'increasing', 'increases'],
        'decrease': ['decrease', 'decreased', 'decreasing', 'decreases'],
        'efficiency': ['efficiency', 'efficient', 'efficiently', 'efficiencies'],
        'renewable': ['renewable', 'renewables', 'renew', 'renewed'],
        'electricity': ['electricity', 'electric', 'electrical', 'electrification'],
        'hydrogen': ['hydrogen'],
        'natural': ['natural', 'naturally', 'nature'],
        'gas': ['gas', 'gases', 'gaseous'],
        'water': ['water', 'waters', 'wastewater'],
        'waste': ['waste', 'wastes', 'wasted', 'wasting'],
        'recycling': ['recycling', 'recycle', 'recycled', 'recyclable'],
        'circular': ['circular', 'circularity'],
        'supply': ['supply', 'supplies', 'supplier', 'suppliers', 'supplying'],
        'chain': ['chain', 'chains'],
        'material': ['material', 'materials', 'raw'],
        'product': ['product', 'products'],
        'customer': ['customer', 'customers'],
        'market': ['market', 'markets', 'marketing'],
        'global': ['global', 'globally', 'globe'],
        'local': ['local', 'locally', 'location', 'locations'],
        'community': ['community', 'communities'],
        'social': ['social', 'socially', 'society', 'societies'],
        'governance': ['governance', 'govern', 'governing', 'government'],
        'compliance': ['compliance', 'comply', 'compliant', 'complying'],
        'regulation': ['regulation', 'regulations', 'regulatory', 'regulate', 'regulated'],
        'standard': ['standard', 'standards', 'standardized'],
        'certification': ['certification', 'certifications', 'certified', 'certify'],
        'initiative': ['initiative', 'initiatives'],
        'project': ['project', 'projects'],
        'program': ['program', 'programs', 'programme', 'programmes'],
        'action': ['action', 'actions'],
        'plan': ['plan', 'plans', 'planned', 'planning'],
        'future': ['future', 'futures'],
        'long-term': ['longterm', 'long'],
        'short-term': ['shortterm', 'short'],
        'transition': ['transition', 'transitions', 'transitioning'],
        'transformation': ['transformation', 'transformations', 'transform', 'transforming'],
        'decarbonization': ['decarbonization', 'decarbonisation', 'decarbonize', 'decarbonise'],
        'net-zero': ['netzero', 'zero', 'neutrality', 'neutral'],
        'scope': ['scope', 'scopes'],
        'footprint': ['footprint', 'footprints'],
        'intensity': ['intensity', 'intensive', 'intensities'],
        'absolute': ['absolute', 'absolutely'],
        'baseline': ['baseline', 'baselines'],
        'pathway': ['pathway', 'pathways', 'path'],
        'roadmap': ['roadmap', 'roadmaps'],
        'milestone': ['milestone', 'milestones'],
        'progress': ['progress', 'progressing', 'progressed'],
        'achievement': ['achievement', 'achievements', 'achieve', 'achieved', 'achieving'],
        'success': ['success', 'successful', 'successfully', 'successes'],
        'value': ['value', 'values', 'valuable', 'valuation'],
        'cost': ['cost', 'costs', 'costly'],
        'benefit': ['benefit', 'benefits', 'beneficial'],
        'stakeholder': ['stakeholder', 'stakeholders'],
        'shareholder': ['shareholder', 'shareholders'],
        'partner': ['partner', 'partners', 'partnership', 'partnerships'],
        'collaboration': ['collaboration', 'collaborations', 'collaborate', 'collaborative'],
        'research': ['research', 'researching', 'researcher', 'researchers'],
        'science': ['science', 'scientific', 'scientist', 'scientists', 'sciencebased'],
    }

    # Create reverse lookup: variant -> representative
    variant_to_rep = {}
    for rep, variants in word_groups.items():
        for v in variants:
            variant_to_rep[v] = rep

    all_words = []
    for text in texts:
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        for w in words:
            if len(w) >= min_word_len and w not in stopwords:
                # Map to representative word if in a group, otherwise keep original
                all_words.append(variant_to_rep.get(w, w))

    return Counter(all_words).most_common(top_n)



# Load texts
print("Loading chunk texts...")
all_texts, texts_by_sentiment, texts_by_commitment = load_all_chunk_texts()
print(f"✅ Loaded {len(all_texts):,} chunks")
print(f"   Opportunity: {len(texts_by_sentiment['opportunity']):,}")
print(f"   Neutral: {len(texts_by_sentiment['neutral']):,}")
print(f"   Risk: {len(texts_by_sentiment['risk']):,}")
print(f"   Commitment YES: {len(texts_by_commitment['yes']):,}")
print(f"   Commitment NO: {len(texts_by_commitment['no']):,}")

# === WORD FREQUENCY ANALYSIS ===
print("\n" + "="*60)
print("TOP WORDS BY CATEGORY")
print("="*60)

freq_all = get_word_frequencies(all_texts, top_n=30)
freq_opp = get_word_frequencies(texts_by_sentiment['opportunity'], top_n=20)
freq_risk = get_word_frequencies(texts_by_sentiment['risk'], top_n=20)
freq_commit = get_word_frequencies(texts_by_commitment['yes'], top_n=20)

print("\n📊 ALL CHUNKS (top 30):")
print(", ".join([f"{w}({c})" for w, c in freq_all]))

print("\n🌱 OPPORTUNITY chunks (top 20):")
print(", ".join([f"{w}({c})" for w, c in freq_opp]))

print("\n⚠️ RISK chunks (top 20):")
print(", ".join([f"{w}({c})" for w, c in freq_risk]))

print("\n🤝 COMMITMENT chunks (top 20):")
print(", ".join([f"{w}({c})" for w, c in freq_commit]))

# === WORD CLOUD ===
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    def make_cloud(ax, texts, title, colormap='viridis'):
        freq = get_word_frequencies(texts, top_n=100)
        if freq:
            wc = WordCloud(width=800, height=400, background_color='white',
                           colormap=colormap, max_words=100)
            wc.generate_from_frequencies(dict(freq))
            ax.imshow(wc, interpolation='bilinear')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

    make_cloud(axes[0, 0], all_texts, '📊 All Climate Chunks', 'viridis')
    make_cloud(axes[0, 1], texts_by_sentiment['opportunity'],
               '🌱 Opportunity Sentiment', 'Greens')
    make_cloud(axes[1, 0], texts_by_sentiment['risk'],
               '⚠️ Risk Sentiment', 'Reds')
    make_cloud(axes[1, 1], texts_by_commitment['yes'],
               '🤝 Commitment Language', 'Blues')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/wordclouds.png', dpi=150, bbox_inches='tight')
    print(f"\n✅ Saved: wordclouds.png")
    plt.show()

except ImportError:
    print("\n⚠️ wordcloud not installed. Run: pip install wordcloud")

# === FREQUENCY BAR CHART (alternative to word cloud) ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))


def plot_freq_bars(ax, freqs, title, color):
    words, counts = zip(*freqs[:15]) if freqs else ([], [])
    y_pos = range(len(words))
    ax.barh(y_pos, counts, color=color, alpha=0.7, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')


plot_freq_bars(axes[0, 0], freq_all[:15],
               '📊 All Chunks - Top Words', '#1976D2')
plot_freq_bars(axes[0, 1], freq_opp[:15],
               '🌱 Opportunity - Top Words', '#4CAF50')
plot_freq_bars(axes[1, 0], freq_risk[:15], '⚠️ Risk - Top Words', '#F44336')
plot_freq_bars(axes[1, 1], freq_commit[:15],
               '🤝 Commitment - Top Words', '#7B1FA2')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/word_frequencies.png', dpi=150, bbox_inches='tight')
print(f"✅ Saved: word_frequencies.png")
plt.show()

## 🎯 Net-Zero Analysis

**Extended Analysis:** Distinguishes between general climate talk and specific net-zero/reduction commitments.

**New Metrics:**
- **Net-Zero %**: Percentage of report focused on emissions reduction targets
- **The Gap**: Climate talk minus net-zero focus (shows "greenwashing" risk)
- **Quality Comparison**: Are net-zero chunks more specific/committed than general climate content?

**Key Insight:** This reveals whether companies are just talking about sustainability broadly or making concrete reduction commitments.

**Additional Outputs:**
- `n0_funnel.png`: Total → Climate → Net-Zero (the narrowing funnel)
- `n0_gap_analysis.png`: Shows gap between climate talk and net-zero action
- `n0_quality_comparison.png`: Specificity/commitment of net-zero vs general content
- `n0_per_company*.png`: Company-level net-zero focus

In [ ]:
# =============================================================================
# NET-ZERO ANALYSIS SECTION
# =============================================================================

def load_with_netzero():
    """Load data including netzero metrics."""
    data_path = Path(DATA_DIR)
    bert_files = [f for f in data_path.glob('*_bert.json')
                  if not any(ex.lower() in f.name.lower() for ex in EXCLUDE_PATTERNS)]

    rows = []
    for fp in bert_files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
            chunks = data.get('chunks', [])
            if not chunks:
                continue

            n = len(chunks)
            detector = np.array([c.get('detector_score', 0) for c in chunks])
            weights = detector if detector.sum() > 0 else np.ones(n)

            # Netzero metrics
            n0_labels = [c.get('netzero_label', 'none') for c in chunks]
            n0_count = sum(1 for l in n0_labels if l == 'reduction')
            n0_pct = n0_count / n * 100 if n > 0 else 0

            # Weighted n0 score (confidence-weighted)
            n0_scores = [c.get('netzero_score', 0) if c.get('netzero_label') == 'reduction' else 0
                         for c in chunks]
            n0_weighted = np.average(
                n0_scores, weights=weights) if weights.sum() > 0 else 0

            # Sentiment/specificity/commitment for N0 vs non-N0 chunks
            n0_chunks = [c for c in chunks if c.get(
                'netzero_label') == 'reduction']
            non_n0_chunks = [c for c in chunks if c.get(
                'netzero_label') != 'reduction']

            def safe_mean(lst, key, default=0):
                vals = [c.get(key, default) for c in lst]
                return np.mean(vals) if vals else default

            spec_labels = [c.get('specificity_label', 'non') for c in chunks]
            sent_labels = [c.get('sentiment_label', 'neutral') for c in chunks]
            comm_labels = [c.get('commitment_label', 'no') for c in chunks]

            opp_weights = sum(w for w, l in zip(
                weights, sent_labels) if l == 'opportunity')
            risk_weights = sum(w for w, l in zip(
                weights, sent_labels) if l == 'risk')
            neutral_weights = sum(w for w, l in zip(
                weights, sent_labels) if l == 'neutral')
            total_weights = weights.sum()

            rows.append({
                'company': data.get('company', 'Unknown'),
                'company_id': str(data.get('company_id', 'unknown')),
                'year': int(data.get('year', 0)),
                'is_translated': data.get('source', 'original') == 'translated',
                # Before climate filter
                'total_chunks': data.get('total_chunks', 0),
                'climate_chunks': data.get('climate_chunks', n),
                'climate_pct': data.get('kept_percentage', 100.0),
                'netzero_count': n0_count,
                'netzero_pct': n0_pct,
                'netzero_weighted': n0_weighted,
                # N0 subset metrics
                'n0_specificity': safe_mean(n0_chunks, 'specificity_score'),
                'n0_commitment': safe_mean(n0_chunks, 'commitment_score'),
                'non_n0_specificity': safe_mean(non_n0_chunks, 'specificity_score'),
                'non_n0_commitment': safe_mean(non_n0_chunks, 'commitment_score'),
                # Standard metrics
                'specificity_w': np.average([c.get('specificity_score', 0) for c in chunks], weights=weights),
                'commitment_w': np.average([c.get('commitment_score', 0) for c in chunks], weights=weights),
                'opportunity_pct': (opp_weights / total_weights * 100) if total_weights > 0 else 0,
                'risk_pct': (risk_weights / total_weights * 100) if total_weights > 0 else 0,
                'neutral_pct': (neutral_weights / total_weights * 100) if total_weights > 0 else 0,
            })
        except Exception as e:
            pass

    df = pd.DataFrame(rows)
    if EXCLUDE_2025:
        df = df[df['year'] < 2025]

    # Aggregate to company-year
    agg_dict = {
        'company': 'first', 'is_translated': 'any',
        'total_chunks': 'sum', 'climate_chunks': 'sum', 'netzero_count': 'sum',
        'climate_pct': 'mean', 'netzero_pct': 'mean', 'netzero_weighted': 'mean',
        'n0_specificity': 'mean', 'n0_commitment': 'mean',
        'non_n0_specificity': 'mean', 'non_n0_commitment': 'mean',
        'specificity_w': 'mean', 'commitment_w': 'mean',
        'opportunity_pct': 'mean', 'risk_pct': 'mean', 'neutral_pct': 'mean',
    }
    cy = df.groupby(['company_id', 'year']).agg(agg_dict).reset_index()

    # Derived metrics
    cy['talk_score'] = ((cy['climate_pct']/100)*0.20 +
                        cy['specificity_w']*0.40 + cy['commitment_w']*0.40) * 100

    # N0-focused talk score (emphasizes net-zero content)
    cy['n0_talk_score'] = ((cy['netzero_pct']/100)*0.30 +
                           cy['n0_specificity']*0.35 + cy['n0_commitment']*0.35) * 100

    return df, cy


# Reload with netzero data
print("Loading data with netzero metrics...")
report_df, cy_df = load_with_netzero()
print(
    f"✅ Loaded: {len(report_df)} reports, {cy_df['company'].nunique()} companies")

# Check if netzero data exists
has_n0 = cy_df['netzero_count'].sum() > 0
if not has_n0:
    print("⚠️ No netzero data found. Run analyze_netzero() on your PDFs first.")
else:
    print(f"   Net-zero chunks: {int(cy_df['netzero_count'].sum()):,}")


# =============================================================================
# N0 PLOT 1: THE FUNNEL - Most Important Visualization
# =============================================================================
# Shows: Total → Climate → Net-Zero (the narrowing of focus)

if has_n0:
    yearly_funnel = cy_df.groupby('year').agg({
        'total_chunks': 'sum',
        'climate_chunks': 'sum',
        'netzero_count': 'sum',
    }).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Absolute numbers (stacked bar)
    ax1 = axes[0]
    years = yearly_funnel['year']

    # Calculate the "other" portions
    non_climate = yearly_funnel['total_chunks'] - \
        yearly_funnel['climate_chunks']
    climate_non_n0 = yearly_funnel['climate_chunks'] - \
        yearly_funnel['netzero_count']
    n0 = yearly_funnel['netzero_count']

    ax1.bar(years, non_climate, label='Non-Climate',
            color='#E0E0E0', edgecolor='white')
    ax1.bar(years, climate_non_n0, bottom=non_climate, label='Climate (General)',
            color='#90CAF9', edgecolor='white')
    ax1.bar(years, n0, bottom=non_climate + climate_non_n0, label='Net-Zero Focus',
            color='#1565C0', edgecolor='white')

    ax1.set_xlabel('Year', fontsize=12)
    ax1.set_ylabel('Number of Chunks', fontsize=12)
    ax1.set_title('Content Composition: The Funnel',
                  fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3, axis='y')

    # Right: Percentages (more useful for presentation)
    ax2 = axes[1]
    total = yearly_funnel['total_chunks']
    climate_pct = yearly_funnel['climate_chunks'] / total * 100
    n0_pct_of_total = yearly_funnel['netzero_count'] / total * 100
    n0_pct_of_climate = yearly_funnel['netzero_count'] / \
        yearly_funnel['climate_chunks'] * 100

    ax2.plot(years, climate_pct, 'b-o', lw=3,
             ms=10, label='Climate % of Total')
    ax2.plot(years, n0_pct_of_total, 'g-s', lw=3,
             ms=10, label='Net-Zero % of Total')
    ax2.plot(years, n0_pct_of_climate, 'r-^', lw=2, ms=8, alpha=0.7,
             label='Net-Zero % of Climate', linestyle='--')

    ax2.set_xlabel('Year', fontsize=12)
    ax2.set_ylabel('Percentage', fontsize=12)
    ax2.set_title('Focus Ratios Over Time', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper left', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, max(climate_pct.max() * 1.1, 100))

    save('n0_funnel.png')


# =============================================================================
# N0 PLOT 2: STACKED AREA - Content Evolution
# =============================================================================

if has_n0:
    yearly_funnel = cy_df.groupby('year').agg({
        'total_chunks': 'sum',
        'climate_chunks': 'sum',
        'netzero_count': 'sum',
    }).reset_index()

    fig, ax = plt.subplots(figsize=(12, 6))

    years = yearly_funnel['year']
    total = yearly_funnel['total_chunks']

    # As percentages of total
    non_climate_pct = (total - yearly_funnel['climate_chunks']) / total * 100
    climate_general_pct = (
        yearly_funnel['climate_chunks'] - yearly_funnel['netzero_count']) / total * 100
    n0_pct = yearly_funnel['netzero_count'] / total * 100

    ax.fill_between(years, 0, n0_pct, color='#1565C0',
                    alpha=0.9, label='Net-Zero Focus')
    ax.fill_between(years, n0_pct, n0_pct + climate_general_pct,
                    color='#64B5F6', alpha=0.7, label='Climate (General)')
    ax.fill_between(years, n0_pct + climate_general_pct, 100,
                    color='#E0E0E0', alpha=0.5, label='Non-Climate')

    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('% of Report Content', fontsize=12)
    ax.set_title('Evolution of Report Content Focus',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)

    # Add annotations for key percentages
    latest = yearly_funnel.iloc[-1]
    latest_n0_pct = latest['netzero_count'] / latest['total_chunks'] * 100
    ax.annotate(f'{latest_n0_pct:.1f}%\nNet-Zero',
                xy=(years.iloc[-1], latest_n0_pct/2),
                fontsize=10, ha='center', color='white', fontweight='bold')

    save('n0_stacked_evolution.png')


# =============================================================================
# N0 PLOT 3: N0 vs GENERAL CLIMATE - Quality Comparison
# =============================================================================
# Key insight: Are net-zero chunks MORE specific/committed than general climate?

if has_n0:
    yearly = cy_df.groupby('year').agg({
        'specificity_w': 'mean',
        'commitment_w': 'mean',
        'n0_specificity': 'mean',
        'n0_commitment': 'mean',
        'non_n0_specificity': 'mean',
        'non_n0_commitment': 'mean',
    }).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Specificity comparison
    ax1 = axes[0]
    ax1.plot(yearly['year'], yearly['n0_specificity']*100, 'b-o', lw=3, ms=9,
             label='Net-Zero Chunks')
    ax1.plot(yearly['year'], yearly['non_n0_specificity']*100, 'gray', ls='--',
             marker='s', lw=2, ms=7, alpha=0.7, label='General Climate Chunks')
    ax1.set_xlabel('Year', fontsize=12)
    ax1.set_ylabel('Specificity Score', fontsize=12)
    ax1.set_title('Specificity: Net-Zero vs General Climate',
                  fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 100)

    # Commitment comparison
    ax2 = axes[1]
    ax2.plot(yearly['year'], yearly['n0_commitment']*100, 'g-o', lw=3, ms=9,
             label='Net-Zero Chunks')
    ax2.plot(yearly['year'], yearly['non_n0_commitment']*100, 'gray', ls='--',
             marker='s', lw=2, ms=7, alpha=0.7, label='General Climate Chunks')
    ax2.set_xlabel('Year', fontsize=12)
    ax2.set_ylabel('Commitment Score', fontsize=12)
    ax2.set_title('Commitment: Net-Zero vs General Climate',
                  fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 100)

    save('n0_quality_comparison.png')


# =============================================================================
# N0 PLOT 4: PER-COMPANY NET-ZERO FOCUS
# =============================================================================

if has_n0:
    company_n0 = cy_df.groupby('company').agg({
        'netzero_pct': ['mean', 'last'],
        'climate_pct': 'mean',
        'netzero_count': 'sum',
        'climate_chunks': 'sum',
    }).reset_index()
    company_n0.columns = ['company', 'n0_pct_mean', 'n0_pct_latest',
                          'climate_pct', 'n0_total', 'climate_total']
    company_n0['n0_of_climate'] = company_n0['n0_total'] / \
        company_n0['climate_total'] * 100
    company_n0 = company_n0.sort_values('n0_pct_latest')

    fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(company_n0)*0.4)))

    # Left: N0 % of total
    ax1 = axes[0]
    colors = plt.cm.Blues(
        company_n0['n0_pct_latest'] / max(company_n0['n0_pct_latest'].max(), 1))
    ax1.barh(company_n0['company'], company_n0['n0_pct_latest'],
             color=colors, edgecolor='white')
    for i, (_, r) in enumerate(company_n0.iterrows()):
        ax1.text(r['n0_pct_latest']+0.3, i,
                 f'{r["n0_pct_latest"]:.1f}%', va='center', fontsize=9)
    ax1.set_xlabel('Net-Zero % of Report (Latest Year)')
    ax1.set_title('Net-Zero Focus by Company', fontweight='bold')
    ax1.set_xlim(0, company_n0['n0_pct_latest'].max() * 1.2)

    # Right: N0 as % of climate content
    ax2 = axes[1]
    colors2 = plt.cm.Greens(
        company_n0['n0_of_climate'] / max(company_n0['n0_of_climate'].max(), 1))
    ax2.barh(company_n0['company'], company_n0['n0_of_climate'],
             color=colors2, edgecolor='white')
    for i, (_, r) in enumerate(company_n0.iterrows()):
        ax2.text(r['n0_of_climate']+0.5, i,
                 f'{r["n0_of_climate"]:.1f}%', va='center', fontsize=9)
    ax2.set_xlabel('Net-Zero % of Climate Content')
    ax2.set_title('Net-Zero Depth (of Climate Talk)', fontweight='bold')
    ax2.set_xlim(0, company_n0['n0_of_climate'].max() * 1.2)

    save('n0_per_company.png')


# =============================================================================
# N0 PLOT 5: COMBINED MAIN SLIDE WITH N0
# =============================================================================

if has_n0:
    yearly = cy_df.groupby('year').agg({
        'climate_pct': 'mean',
        'netzero_pct': 'mean',
        'specificity_w': 'mean',
        'commitment_w': 'mean',
    }).reset_index()

    fig, ax = plt.subplots(figsize=(12, 7))

    ax.plot(yearly['year'], yearly['climate_pct'], '#7B1FA2', lw=3.5,
            marker='o', ms=11, label='Volume (% climate)')
    ax.plot(yearly['year'], yearly['netzero_pct'], '#E91E63', lw=3.5,
            marker='D', ms=10, label='Net-Zero Focus (%)', linestyle='--')
    ax.plot(yearly['year'], yearly['specificity_w']*100, '#1565C0',
            lw=3.5, marker='s', ms=11, label='Specificity')
    ax.plot(yearly['year'], yearly['commitment_w']*100, '#2E7D32',
            lw=3.5, marker='^', ms=11, label='Commitment')

    ax.set_xlabel('Year', fontsize=14)
    ax.set_ylabel('Score (0-100)', fontsize=14)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.set_title('Climate Reporting in the Steel Industry\n(with Net-Zero Focus)',
                 fontsize=18, fontweight='bold')
    # Move legend to upper left
    ax.legend(loc='upper left', fontsize=11, frameon=True, fancybox=True,
              shadow=True, facecolor='#F5F5F5', edgecolor='gray')

    save('slide_main_with_n0.png')

# =============================================================================
# N0 PLOT 6: THE GAP - Climate Talk vs N0 Action
# =============================================================================

if has_n0:
    yearly = cy_df.groupby('year').agg({
        'climate_pct': 'mean',
        'netzero_pct': 'mean',
    }).reset_index()

    fig, ax = plt.subplots(figsize=(12, 6))

    years = yearly['year']
    climate = yearly['climate_pct']
    n0 = yearly['netzero_pct']
    gap = climate - n0

    # Fill the gap area
    ax.fill_between(years, n0, climate, color='#FFCDD2',
                    alpha=0.7, label='The Gap')
    ax.plot(years, climate, '#7B1FA2', lw=4,
            marker='o', ms=12, label='Climate Talk')
    ax.plot(years, n0, '#1565C0', lw=4, marker='s',
            ms=12, label='Net-Zero Focus')

    # Annotate the gap
    mid_year = years.iloc[len(years)//2]
    mid_climate = climate.iloc[len(years)//2]
    mid_n0 = n0.iloc[len(years)//2]
    ax.annotate('', xy=(mid_year, mid_n0), xytext=(mid_year, mid_climate),
                arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    ax.text(mid_year + 0.3, (mid_climate + mid_n0)/2, f'Gap:\n{mid_climate - mid_n0:.1f}%',
            fontsize=11, color='red', fontweight='bold')

    ax.set_xlabel('Year', fontsize=14)
    ax.set_ylabel('% of Report Content', fontsize=14)
    ax.set_title('The Gap: Climate Talk vs Net-Zero Focus',
                 fontsize=16, fontweight='bold')
    ax.legend(loc='upper left', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(climate.max() * 1.2, 50))

    save('n0_gap_analysis.png')


# =============================================================================
# N0 PLOT 7: PER-COMPANY TRAJECTORIES
# =============================================================================

if has_n0:
    companies = sorted(cy_df['company'].unique())
    n_cols, n_rows = 3, int(np.ceil(len(companies)/3))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(
        5*n_cols, 4*n_rows), squeeze=False)
    axes = axes.flatten()

    for i, company in enumerate(companies):
        ax = axes[i]
        cd = cy_df[cy_df['company'] == company].sort_values('year')

        # Stacked area: N0 (bottom), General Climate (top)
        n0 = cd['netzero_pct']
        climate_general = cd['climate_pct'] - cd['netzero_pct']

        ax.fill_between(cd['year'], 0, n0, color='#1565C0',
                        alpha=0.9, label='Net-Zero')
        ax.fill_between(cd['year'], n0, n0 + climate_general,
                        color='#90CAF9', alpha=0.6, label='Climate (General)')

        ax.set_title(company, fontweight='bold', fontsize=10)
        ax.set_ylim(0, max(cd['climate_pct'].max() * 1.2, 30))
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, loc='upper left')

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Per-Company: Net-Zero vs General Climate Content',
                 fontsize=14, fontweight='bold', y=1.01)
    save('n0_per_company_trajectories.png')


# =============================================================================
# SUMMARY WITH N0
# =============================================================================

if has_n0:
    total_climate = int(cy_df['climate_chunks'].sum())
    total_n0 = int(cy_df['netzero_count'].sum())
    n0_ratio = total_n0 / total_climate * 100 if total_climate > 0 else 0

    latest_year = cy_df['year'].max()
    latest = cy_df[cy_df['year'] == latest_year]
    earliest_year = cy_df['year'].min()
    earliest = cy_df[cy_df['year'] == earliest_year]

    print(f"""
{'='*60}
📊 NET-ZERO ANALYSIS SUMMARY
{'='*60}
Overall:
  Climate chunks: {total_climate:,}
  Net-Zero chunks: {total_n0:,} ({n0_ratio:.1f}% of climate content)

Trends ({earliest_year} → {latest_year}):
  Climate % of reports: {earliest['climate_pct'].mean():.1f}% → {latest['climate_pct'].mean():.1f}%
  Net-Zero % of reports: {earliest['netzero_pct'].mean():.1f}% → {latest['netzero_pct'].mean():.1f}%

Key Insight:
  Only {n0_ratio:.1f}% of climate content specifically addresses
  net-zero/emissions reduction — the rest is general sustainability talk.
{'='*60}
""")

## 📝 Word Frequency Analysis

Extracts the most common terms in different discourse categories to understand thematic patterns.

**Word Grouping Strategy:**
- Groups lexical variants to a single representative (e.g., "emission", "emissions", "emitted" → "emission")
- Filters stopwords and generic company/report terms
- Weight by detector confidence scores

**Categories Analyzed:**
1. All climate chunks
2. Opportunity sentiment chunks
3. Risk sentiment chunks  
4. Commitment language chunks
5. Net-zero specific chunks (if available)

In [ ]:
# =============================================================================
# N0 WORD CLOUD & FREQUENCY ANALYSIS
# =============================================================================

def load_n0_chunk_texts():
    """Load chunk texts separated by netzero label."""
    data_path = Path(DATA_DIR)
    bert_files = [f for f in data_path.glob('*_bert.json')
                  if not any(ex.lower() in f.name.lower() for ex in EXCLUDE_PATTERNS)]

    texts_by_n0 = {'net-zero': [], 'reduction': [], 'none': []}

    for fp in bert_files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)

            year = data.get('year', 0)
            if EXCLUDE_2025 and year >= 2025:
                continue

            for chunk in data.get('chunks', []):
                text = chunk.get('text', '')
                n0_label = chunk.get('netzero_label', 'none')
                if text and n0_label in texts_by_n0:
                    texts_by_n0[n0_label].append(text)
        except:
            pass

    return texts_by_n0


if has_n0:
    print("\n" + "="*60)
    print("NET-ZERO WORD FREQUENCY ANALYSIS")
    print("="*60)

    texts_by_n0 = load_n0_chunk_texts()

    print(f"\nChunk counts by N0 label:")
    for label, texts in texts_by_n0.items():
        print(f"   {label}: {len(texts):,}")

    # Get frequencies
    freq_netzero = get_word_frequencies(texts_by_n0['net-zero'], top_n=25)
    freq_reduction = get_word_frequencies(texts_by_n0['reduction'], top_n=25)
    freq_none = get_word_frequencies(texts_by_n0['none'], top_n=25)

    print("\n🎯 NET-ZERO chunks (top 25):")
    print(", ".join([f"{w}({c})" for w, c in freq_netzero]))

    print("\n📉 REDUCTION chunks (top 25):")
    print(", ".join([f"{w}({c})" for w, c in freq_reduction]))

    print("\n⬜ NONE (climate but not target) chunks (top 25):")
    print(", ".join([f"{w}({c})" for w, c in freq_none]))

    # Word clouds
    try:
        from wordcloud import WordCloud

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        def make_cloud(ax, texts, title, colormap):
            freq = get_word_frequencies(texts, top_n=80)
            if freq:
                wc = WordCloud(width=600, height=400, background_color='white',
                               colormap=colormap, max_words=80)
                wc.generate_from_frequencies(dict(freq))
                ax.imshow(wc, interpolation='bilinear')
            ax.set_title(title, fontsize=14, fontweight='bold')
            ax.axis('off')

        make_cloud(axes[0], texts_by_n0['net-zero'],
                   '🎯 Net-Zero Targets', 'Blues')
        make_cloud(axes[1], texts_by_n0['reduction'],
                   '📉 Reduction Targets', 'Greens')
        make_cloud(axes[2], texts_by_n0['none'],
                   '⬜ Climate (No Target)', 'Greys')

        plt.tight_layout()
        save('n0_wordclouds.png')

    except ImportError:
        print("⚠️ wordcloud not installed")

    # Bar chart comparison
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))

    def plot_freq(ax, freqs, title, color):
        if not freqs:
            return
        words, counts = zip(*freqs[:12])
        y_pos = range(len(words))
        ax.barh(y_pos, counts, color=color, alpha=0.8, edgecolor='white')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words)
        ax.invert_yaxis()
        ax.set_xlabel('Frequency')
        ax.set_title(title, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')

    plot_freq(axes[0], freq_netzero, '🎯 Net-Zero', '#1565C0')
    plot_freq(axes[1], freq_reduction, '📉 Reduction', '#2E7D32')
    plot_freq(axes[2], freq_none, '⬜ No Target', '#757575')

    plt.tight_layout()
    save('n0_word_frequencies.png')



# CHUNK ANALYZER - Inspect Netzero chunks
---------------------------------------------

In [ ]:

# =============================================================================
# CHUNK ANALYZER - Inspect N0 chunks
# =============================================================================

def analyze_n0_chunks(n_samples=10, label_filter=None, random_seed=42,
                      show_scores=True, export_path=None):
    """
    Analyze and inspect net-zero classified chunks.

    Args:
        n_samples: Number of random samples to display (default 10)
        label_filter: 'net-zero', 'reduction', 'none', or None for all
        random_seed: For reproducibility
        show_scores: Show confidence scores
        export_path: If provided, export all matching chunks to this JSON file
    """
    import random
    random.seed(random_seed)

    data_path = Path(DATA_DIR)
    bert_files = [f for f in data_path.glob('*_bert.json')
                  if not any(ex.lower() in f.name.lower() for ex in EXCLUDE_PATTERNS)]

    all_chunks = []

    for fp in bert_files:
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)

            year = data.get('year', 0)
            if EXCLUDE_2025 and year >= 2025:
                continue

            company = data.get('company', 'Unknown')

            for chunk in data.get('chunks', []):
                n0_label = chunk.get('netzero_label', 'none')

                if label_filter and n0_label != label_filter:
                    continue

                all_chunks.append({
                    'company': company,
                    'year': year,
                    # Truncate for display
                    'text': chunk.get('text', '')[:500],
                    'full_text': chunk.get('text', ''),
                    'netzero_label': n0_label,
                    'netzero_score': chunk.get('netzero_score', 0),
                    'detector_score': chunk.get('detector_score', 0),
                    'specificity_label': chunk.get('specificity_label', ''),
                    'commitment_label': chunk.get('commitment_label', ''),
                })
        except:
            pass

    # Export if requested
    if export_path:
        export_data = [{'company': c['company'], 'year': c['year'],
                        'text': c['full_text'], 'netzero_label': c['netzero_label'],
                        'netzero_score': c['netzero_score']} for c in all_chunks]
        with open(export_path, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, ensure_ascii=False, indent=2)
        print(f"✅ Exported {len(export_data)} chunks to {export_path}")

    # Summary stats
    label_counts = Counter(c['netzero_label'] for c in all_chunks)
    print(f"\n{'='*60}")
    print(f"N0 CHUNK ANALYSIS")
    print(f"{'='*60}")
    print(f"Total chunks: {len(all_chunks):,}")
    print(f"Filter: {label_filter or 'all'}")
    print(f"\nLabel distribution:")
    for label, count in sorted(label_counts.items()):
        pct = count / len(all_chunks) * 100
        print(f"   {label}: {count:,} ({pct:.1f}%)")

    # Score distribution for filtered label
    if label_filter:
        scores = [c['netzero_score'] for c in all_chunks]
        print(f"\nConfidence scores for '{label_filter}':")
        print(f"   Mean: {np.mean(scores):.3f}")
        print(f"   Median: {np.median(scores):.3f}")
        print(f"   Min: {np.min(scores):.3f}, Max: {np.max(scores):.3f}")

    # Sample display
    samples = random.sample(all_chunks, min(n_samples, len(all_chunks)))

    print(f"\n{'='*60}")
    print(f"SAMPLE CHUNKS (n={len(samples)})")
    print(f"{'='*60}")

    for i, chunk in enumerate(samples, 1):
        print(f"\n--- [{i}] {chunk['company']} ({chunk['year']}) ---")
        print(f"Label: {chunk['netzero_label']}", end='')
        if show_scores:
            print(f" | Score: {chunk['netzero_score']:.3f} | "
                  f"Spec: {chunk['specificity_label']} | Comm: {chunk['commitment_label']}")
        else:
            print()
        print(f"\n{chunk['text']}")
        if len(chunk['full_text']) > 500:
            print(f"... [truncated, {len(chunk['full_text'])} chars total]")

    return all_chunks


# === USAGE EXAMPLES ===
#
# Browse random net-zero chunks
analyze_n0_chunks(n_samples=10, label_filter='net-zero')
#
# Browse reduction targets
# analyze_n0_chunks(n_samples=10, label_filter='reduction')
#
# Export all net-zero chunks for manual review
# chunks = analyze_n0_chunks(label_filter='net-zero', export_path='n0_netzero_chunks.json')
#
# Export ALL chunks with N0 labels
# chunks = analyze_n0_chunks(export_path='all_n0_analyzed_chunks.json')